# How Machines Learn: A Supervised Learning Lab

This notebook accompanies the browser-based interactive lecture. You will build classification and regression models, keep test data separate from training, compare preprocessing choices, and evaluate generalization.

**Learning goals**
- Create and split labeled datasets.
- Fit classification and regression pipelines with scikit-learn.
- Interpret held-out metrics and a confusion matrix.
- Explain why preprocessing must be fitted on training data only.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons, make_regression
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, mean_squared_error, r2_score)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Ready to explore supervised learning.")

## 1. Classification: learn a curved decision boundary

The two-moons dataset is deliberately not linearly separable. We will reserve a stratified test split before fitting any model.

In [ ]:
X, y = make_moons(n_samples=500, noise=0.28, random_state=RANDOM_STATE)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

print(f"Training examples: {len(X_train)}")
print(f"Test examples:     {len(X_test)}")
plt.figure(figsize=(6, 4))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="coolwarm", s=18, label="Training")
plt.title("Training examples (color = class)")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

## 2. Fit and evaluate logistic regression

A pipeline ensures the scaler is fitted using training data only. The test examples are used only for the final evaluation.

In [ ]:
logistic_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
)
logistic_model.fit(X_train, y_train)
logistic_predictions = logistic_model.predict(X_test)

print(f"Held-out accuracy: {accuracy_score(y_test, logistic_predictions):.3f}")
print("Confusion matrix (rows = actual, columns = predicted):")
print(confusion_matrix(y_test, logistic_predictions))
print(classification_report(y_test, logistic_predictions, zero_division=0))

## 3. Compare with k-nearest neighbors

k-nearest neighbors predicts from nearby training examples. Distance-based methods are sensitive to feature scales, so standardization is especially important.

In [ ]:
knn_model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7))
knn_model.fit(X_train, y_train)
knn_predictions = knn_model.predict(X_test)

comparison = pd.DataFrame({
    "model": ["Logistic regression", "7-nearest neighbors"],
    "test accuracy": [accuracy_score(y_test, logistic_predictions),
                      accuracy_score(y_test, knn_predictions)]
})
comparison

### Inspect the k-nearest-neighbors decision regions

In [ ]:
x_min, x_max = X[:, 0].min() - 0.4, X[:, 0].max() + 0.4
y_min, y_max = X[:, 1].min() - 0.4, X[:, 1].max() + 0.4
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 220),
                     np.linspace(y_min, y_max, 220))
grid = np.c_[xx.ravel(), yy.ravel()]
regions = knn_model.predict(grid).reshape(xx.shape)

plt.figure(figsize=(7, 5))
plt.contourf(xx, yy, regions, alpha=0.2, cmap="coolwarm")
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="coolwarm", s=16, alpha=0.55)
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap="coolwarm", s=30,
            edgecolors="black", linewidths=0.5, label="Held-out test examples")
plt.title("k-nearest-neighbors decision regions")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.show()

## 4. Regression: predict a continuous value

The next dataset contains a noisy relationship between input features and a real-valued target. Compare the model's predictions with the held-out targets using RMSE and R².

In [ ]:
X_reg, y_reg = make_regression(
    n_samples=500, n_features=4, n_informative=3,
    noise=18.0, random_state=RANDOM_STATE
)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.25, random_state=RANDOM_STATE
)

regression_model = make_pipeline(StandardScaler(), LinearRegression())
regression_model.fit(Xr_train, yr_train)
regression_predictions = regression_model.predict(Xr_test)

rmse = np.sqrt(mean_squared_error(yr_test, regression_predictions))
r_squared = r2_score(yr_test, regression_predictions)
print(f"Held-out RMSE: {rmse:.2f} (lower is better)")
print(f"Held-out R²:   {r_squared:.3f} (closer to 1 is better)")

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(yr_test, regression_predictions, alpha=0.7)
limits = [min(yr_test.min(), regression_predictions.min()),
          max(yr_test.max(), regression_predictions.max())]
plt.plot(limits, limits, "--", color="black", label="Perfect predictions")
plt.xlabel("Actual target")
plt.ylabel("Predicted target")
plt.title("Regression: held-out predictions")
plt.legend()
plt.show()

## 5. Experiments and reflection

Try changing the settings and answer these questions:

1. How does the test score change as the number of training examples increases?
2. Which model captures the curved two-moons boundary better? What happens if you change `n_neighbors`?
3. Why should a scaler be fitted on training data rather than on the full dataset?
4. Would accuracy alone be a good measure if one class were much rarer than the other?
5. What does a negative R² tell you about the regression model?

**Practice:** Change the noise level in `make_moons`, rerun the split and both classifiers, and record how the confusion matrix changes. Then try another dataset from `sklearn.datasets`.